<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left">  Reason → Act → Observe Loop </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Show how agents perform **multi-step reasoning** using the Reason–Act–Observe loop.

### What you will learn:
- How to implement the reasoning loop
- How an agent solves a multi-step task (get key → open door → reach goal)
- How to print a clear execution trace
- How stopping conditions work

**Why this matters:**  
Real AI agents (like AutoGPT or robotic agents) don't just act randomly. They **think first**, then act, then observe the result and repeat.

## 1. What is the Reason → Act → Observe Loop?

- **Reason**: The agent looks at the current situation and decides what to do next. It also explains its thinking in plain English.
- **Act**: The agent performs **one** action (move up, pick up key, open door, etc.).
- **Observe**: The agent receives new information from the environment.

This loop continues until the goal is reached or a stopping condition is met.

**Benefits:**
- Makes the agent's decisions transparent and easy to debug.
- Perfect for teaching multi-step reasoning in Agentic AI.

## 2. The GridWorld Environment

We use a simple grid world:
- `A` = Agent (you)
- `K` = Key
- `D` = Locked door
- `G` = Goal
- `#` = Wall
- ` ` = Empty space

**Task:** The agent must pick up the key, open the locked door, then reach the goal.

In [1]:
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from collections import deque

@dataclass
class GridState:
    grid: List[List[str]]
    agent_pos: Tuple[int, int]
    has_key: bool = False
    door_open: bool = False

class GridWorld:
    def __init__(self, grid: List[str]):
        self.grid_rows = [list(row) for row in grid]
        self.h = len(self.grid_rows)
        self.w = len(self.grid_rows[0]) if self.h > 0 else 0

        self.agent_pos = None
        self.key_pos = None
        self.door_pos = None
        self.goal_pos = None

        for r in range(self.h):
            for c in range(self.w):
                if self.grid_rows[r][c] == 'A':
                    self.agent_pos = (r, c)
                    self.grid_rows[r][c] = ' '
                elif self.grid_rows[r][c] == 'K':
                    self.key_pos = (r, c)
                    self.grid_rows[r][c] = ' '
                elif self.grid_rows[r][c] == 'D':
                    self.door_pos = (r, c)
                elif self.grid_rows[r][c] == 'G':
                    self.goal_pos = (r, c)

        self.state = GridState(grid=self.grid_rows, agent_pos=self.agent_pos)

    def in_bounds(self, pos):
        r, c = pos
        return 0 <= r < self.h and 0 <= c < self.w

    def is_passable(self, pos):
        r, c = pos
        if self.grid_rows[r][c] == '#':
            return False
        if self.door_pos and pos == self.door_pos and not self.state.door_open:
            return False
        return True

    def step(self, action: str):
        r, c = self.state.agent_pos
        reward = 0.0
        info = {}

        if action in ('up', 'down', 'left', 'right'):
            dr, dc = {'up': (-1,0), 'down':(1,0), 'left':(0,-1), 'right':(0,1)}[action]
            new_pos = (r + dr, c + dc)
            if self.in_bounds(new_pos) and self.is_passable(new_pos):
                self.state.agent_pos = new_pos
                info['moved'] = True
            else:
                info['moved'] = False

        elif action == 'pickup':
            if self.state.agent_pos == self.key_pos and not self.state.has_key:
                self.state.has_key = True
                reward = 1.0
                info['picked'] = True
            else:
                info['picked'] = False

        elif action == 'open':
            if (self.door_pos and self.state.agent_pos == self.door_pos and
                self.state.has_key and not self.state.door_open):
                self.state.door_open = True
                reward = 1.0
                info['opened'] = True
            else:
                info['opened'] = False

        obs = {
            'agent_pos': self.state.agent_pos,
            'has_key': self.state.has_key,
            'door_open': self.state.door_open,
            'key_pos': self.key_pos,
            'door_pos': self.door_pos,
            'goal_pos': self.goal_pos,
            'cell': self._cell_content(self.state.agent_pos)
        }
        done = (self.state.agent_pos == self.goal_pos)
        return obs, reward, done, info

    def _cell_content(self, pos):
        r, c = pos
        if self.key_pos and pos == self.key_pos and not self.state.has_key:
            return 'K'
        if self.door_pos and pos == self.door_pos and not self.state.door_open:
            return 'D'
        if self.goal_pos and pos == self.goal_pos:
            return 'G'
        return self.grid_rows[r][c]

    def render(self):
        grid_copy = [row.copy() for row in self.grid_rows]
        ra, ca = self.state.agent_pos
        grid_copy[ra][ca] = 'A'
        if not self.state.has_key and self.key_pos:
            rk, ck = self.key_pos
            grid_copy[rk][ck] = 'K'
        if self.door_pos:
            rd, cd = self.door_pos
            grid_copy[rd][cd] = ' ' if self.state.door_open else 'D'
        if self.goal_pos:
            rg, cg = self.goal_pos
            grid_copy[rg][cg] = 'G'

        for row in grid_copy:
            print(''.join(row))
        print(f"Has Key: {self.state.has_key} | Door Open: {self.state.door_open}\n")

## 3. Pathfinding with BFS

We use **BFS** ([Breadth-First Search](https://www.khanacademy.org/computing/computer-science/algorithms/breadth-first-search/a/the-breadth-first-search-algorithm)) inside the Reason step to help the agent plan the shortest path.

**Do we need complicated things?**  
No. You can remove BFS later and replace it with simple movement.  
But keeping BFS makes the agent smarter and this demo more realistic.

In [2]:
def bfs_shortest_path(grid_world, start, goal, consider_door_open):
    if start is None or goal is None:
        return None
    visited = set()
    q = deque([(start, [start])])
    visited.add(start)

    while q:
        pos, path = q.popleft()
        if pos == goal:
            return path
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            new = (pos[0] + dr, pos[1] + dc)
            if not grid_world.in_bounds(new):
                continue
            if grid_world.door_pos and new == grid_world.door_pos and not consider_door_open:
                continue
            if grid_world.grid_rows[new[0]][new[1]] == '#':
                continue
            if new in visited:
                continue
            visited.add(new)
            q.append((new, path + [new]))
    return None

## 4. The Agent Class (Reason → Act → Observe)

This is the core of the demo.

In [5]:
class Agent:
    def __init__(self, env, max_steps=100):
        self.env = env
        self.max_steps = max_steps
        self.trace = []
        self.steps = 0

    def reason(self, obs):
        agent_pos = obs['agent_pos']
        has_key = obs['has_key']
        door_open = obs['door_open']
        key_pos = obs['key_pos']
        door_pos = obs['door_pos']
        goal_pos = obs['goal_pos']

        thought_lines = []

        if not has_key:
            thought_lines.append("I don't have the key yet.")
            if agent_pos == key_pos:
                action = 'pickup'
                thought_lines.append("I'm standing on the key → picking it up now.")
            else:
                path = bfs_shortest_path(self.env, agent_pos, key_pos, consider_door_open=False)
                if path and len(path) > 1:
                    next_pos = path[1]
                    dr = next_pos[0] - agent_pos[0]
                    dc = next_pos[1] - agent_pos[1]
                    move_map = {(-1,0):'up', (1,0):'down', (0,-1):'left', (0,1):'right'}
                    action = move_map.get((dr, dc), 'up')
                    thought_lines.append(f"Planning to go to key. Next move: {action}")
                else:
                    action = 'up'
                    thought_lines.append("No clear path to key. Moving up as fallback.")

        elif not door_open:
            thought_lines.append("I have the key, but the door is still closed.")
            if agent_pos == door_pos:
                action = 'open'
                thought_lines.append("I'm at the door with the key → opening it now.")
            else:
                path = bfs_shortest_path(self.env, agent_pos, door_pos, consider_door_open=False)
                if path and len(path) > 1:
                    next_pos = path[1]
                    dr = next_pos[0] - agent_pos[0]
                    dc = next_pos[1] - agent_pos[1]
                    move_map = {(-1,0):'up', (1,0):'down', (0,-1):'left', (0,1):'right'}
                    action = move_map.get((dr, dc), 'up')
                    thought_lines.append(f"Planning to go to door. Next move: {action}")
                else:
                    action = 'up'
                    thought_lines.append("No clear path to door. Moving up as fallback.")

        else:
            thought_lines.append("I have the key and the door is open. Going to the goal.")
            if agent_pos == goal_pos:
                action = None
                thought_lines.append("I'm already at the goal!")
            else:
                path = bfs_shortest_path(self.env, agent_pos, goal_pos, consider_door_open=True)
                if path and len(path) > 1:
                    next_pos = path[1]
                    dr = next_pos[0] - agent_pos[0]
                    dc = next_pos[1] - agent_pos[1]
                    move_map = {(-1,0):'up', (1,0):'down', (0,-1):'left', (0,1):'right'}
                    action = move_map.get((dr, dc), 'up')
                    thought_lines.append(f"Planning to go to goal. Next move: {action}")
                else:
                    action = 'up'
                    thought_lines.append("No clear path to goal. Moving up as fallback.")

        thought = " | ".join(thought_lines)
        return {'thought': thought, 'action': action}

    def act(self, action):
        if action is None:
            return None, 0.0, False, {'noop': True}
        return self.env.step(action)

    def run(self):
        # Initial observation - FIXED
        obs = {
            'agent_pos': self.env.state.agent_pos,
            'has_key': self.env.state.has_key,
            'door_open': self.env.state.door_open,
            'key_pos': self.env.key_pos,      # ← Fixed: use env.key_pos
            'door_pos': self.env.door_pos,    # ← Fixed: use env.door_pos
            'goal_pos': self.env.goal_pos,    # ← Fixed: use env.goal_pos
            'cell': self.env._cell_content(self.env.state.agent_pos)
        }

        done = False
        last_positions = deque(maxlen=6)

        print("🚀 Starting Reason → Act → Observe Loop...\n")

        while not done and self.steps < self.max_steps:
            self.steps += 1
            reasoning = self.reason(obs)
            action = reasoning['action']
            thought = reasoning['thought']

            new_obs, reward, done, info = self.act(action)

            if new_obs is None:
                new_obs = {
                    'agent_pos': self.env.state.agent_pos,
                    'has_key': self.env.state.has_key,
                    'door_open': self.env.state.door_open,
                    'key_pos': self.env.key_pos,
                    'door_pos': self.env.door_pos,
                    'goal_pos': self.env.goal_pos,
                    'cell': self.env._cell_content(self.env.state.agent_pos)
                }

            # Save to trace
            self.trace.append({
                'step': self.steps,
                'thought': thought,
                'action': action,
                'observation': new_obs,
                'info': info
            })

            # Print step information
            print(f"Step {self.steps}:")
            print(f"  Thought: {thought}")
            print(f"  Action : {action}")
            print(f"  Info   : {info}")
            self.env.render()
            print("-" * 60)

            # Check if stuck
            last_positions.append(self.env.state.agent_pos)
            if len(last_positions) == last_positions.maxlen and len(set(last_positions)) == 1:
                print("Agent is stuck (no movement). Stopping.")
                break

            obs = new_obs

        if done:
            print(f" Goal reached in {self.steps} steps!")
        else:
            print("Stopped (max steps reached or agent stuck).")

        return self.trace

## 5. Running the Demo

Now we create the environment and run the agent.

In [7]:
# Create the grid (all rows must be same length)
grid = [
    "##########",
    "#A       #",
    "#  #     #",
    "#  K  D  #",
    "#  #     #",
    "#       G#",
    "##########"
]

env = GridWorld(grid)
print("Initial Environment:")
env.render()

agent = Agent(env, max_steps=100)
trace = agent.run()

# Summary
print("\n=== EXECUTION TRACE SUMMARY ===")
for entry in trace:
    print(f"Step {entry['step']}: Action = {entry['action']} | Thought = {entry['thought'].split(' | ')[0]}")

Initial Environment:
##########
#A       #
#  #     #
#  K  D  #
#  #     #
#       G#
##########
Has Key: False | Door Open: False

🚀 Starting Reason → Act → Observe Loop...

Step 1:
  Thought: I don't have the key yet. | Planning to go to key. Next move: down
  Action : down
  Info   : {'moved': True}
##########
#        #
#A #     #
#  K  D  #
#  #     #
#       G#
##########
Has Key: False | Door Open: False

------------------------------------------------------------
Step 2:
  Thought: I don't have the key yet. | Planning to go to key. Next move: down
  Action : down
  Info   : {'moved': True}
##########
#        #
#  #     #
#A K  D  #
#  #     #
#       G#
##########
Has Key: False | Door Open: False

------------------------------------------------------------
Step 3:
  Thought: I don't have the key yet. | Planning to go to key. Next move: right
  Action : right
  Info   : {'moved': True}
##########
#        #
#  #     #
# AK  D  #
#  #     #
#       G#
##########
Has Key: Fal

# Simple Demo: Reason → Act → Observe Loop

**No complicated algorithms** — just simple rules.

## 1. The GridWorld Environment

This is a simple grid (like a small room) where the agent moves.
- `A` = Agent
- `K` = Key to pick up
- `D` = Locked door
- `G` = Goal
- `#` = Wall (cannot pass)

In [8]:
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from collections import deque

@dataclass
class GridState:
    grid: List[List[str]]
    agent_pos: Tuple[int, int]
    has_key: bool = False
    door_open: bool = False

class GridWorld:
    """Simple GridWorld for teaching"""
    def __init__(self, grid: List[str]):
        self.grid_rows = [list(row) for row in grid]
        self.h = len(self.grid_rows)
        self.w = len(self.grid_rows[0]) if self.h > 0 else 0

        self.agent_pos = None
        self.key_pos = None
        self.door_pos = None
        self.goal_pos = None

        for r in range(self.h):
            for c in range(self.w):
                ch = self.grid_rows[r][c]
                if ch == 'A':
                    self.agent_pos = (r, c)
                    self.grid_rows[r][c] = ' '
                elif ch == 'K':
                    self.key_pos = (r, c)
                    self.grid_rows[r][c] = ' '
                elif ch == 'D':
                    self.door_pos = (r, c)
                elif ch == 'G':
                    self.goal_pos = (r, c)

        self.state = GridState(grid=self.grid_rows, agent_pos=self.agent_pos)

    def in_bounds(self, pos: Tuple[int, int]) -> bool:
        r, c = pos
        return 0 <= r < self.h and 0 <= c < self.w

    def is_passable(self, pos: Tuple[int, int]) -> bool:
        r, c = pos
        if self.grid_rows[r][c] == '#':
            return False
        if self.door_pos and pos == self.door_pos and not self.state.door_open:
            return False
        return True

    def step(self, action: str):
        r, c = self.state.agent_pos
        reward = 0.0
        info = {}

        if action in ('up', 'down', 'left', 'right'):
            dr, dc = {'up': (-1,0), 'down':(1,0), 'left':(0,-1), 'right':(0,1)}[action]
            new_pos = (r + dr, c + dc)
            if self.in_bounds(new_pos) and self.is_passable(new_pos):
                self.state.agent_pos = new_pos
                info['moved'] = True
            else:
                info['moved'] = False

        elif action == 'pickup':
            if self.state.agent_pos == self.key_pos and not self.state.has_key:
                self.state.has_key = True
                reward = 1.0
                info['picked'] = True
            else:
                info['picked'] = False

        elif action == 'open':
            if (self.door_pos and self.state.agent_pos == self.door_pos and
                self.state.has_key and not self.state.door_open):
                self.state.door_open = True
                reward = 1.0
                info['opened'] = True
            else:
                info['opened'] = False

        obs = {
            'agent_pos': self.state.agent_pos,
            'has_key': self.state.has_key,
            'door_open': self.state.door_open,
            'key_pos': self.key_pos,
            'door_pos': self.door_pos,
            'goal_pos': self.goal_pos,
            'cell': self._cell_content(self.state.agent_pos)
        }
        done = (self.state.agent_pos == self.goal_pos)
        return obs, reward, done, info

    def _cell_content(self, pos):
        r, c = pos
        if self.key_pos and pos == self.key_pos and not self.state.has_key:
            return 'K'
        if self.door_pos and pos == self.door_pos and not self.state.door_open:
            return 'D'
        if self.goal_pos and pos == self.goal_pos:
            return 'G'
        return self.grid_rows[r][c]

    def render(self):
        grid_copy = [row.copy() for row in self.grid_rows]
        ra, ca = self.state.agent_pos
        grid_copy[ra][ca] = 'A'
        if not self.state.has_key and self.key_pos:
            rk, ck = self.key_pos
            grid_copy[rk][ck] = 'K'
        if self.door_pos:
            rd, cd = self.door_pos
            grid_copy[rd][cd] = ' ' if self.state.door_open else 'D'
        if self.goal_pos:
            rg, cg = self.goal_pos
            grid_copy[rg][cg] = 'G'

        for row in grid_copy:
            print(''.join(row))
        print(f"Has Key: {self.state.has_key} | Door Open: {self.state.door_open}\n")

In [9]:
class Agent:
    """Simple Agent with Reason → Act → Observe loop (No BFS)"""

    def __init__(self, env, max_steps=50):
        self.env = env
        self.max_steps = max_steps
        self.trace = []
        self.steps = 0

    def reason(self, obs):
        """Simple rule-based reasoning - very easy for students to understand"""
        agent_pos = obs['agent_pos']
        has_key = obs['has_key']
        door_open = obs['door_open']
        key_pos = obs['key_pos']
        door_pos = obs['door_pos']
        goal_pos = obs['goal_pos']

        thought_lines = []

        if not has_key:
            thought_lines.append("I don't have the key yet.")
            if agent_pos == key_pos:
                action = 'pickup'
                thought_lines.append("I am on the key! I will pick it up.")
            else:
                # Simple movement: try to move toward the key (basic direction)
                ar, ac = agent_pos
                kr, kc = key_pos
                if ar < kr:
                    action = 'down'
                elif ar > kr:
                    action = 'up'
                elif ac < kc:
                    action = 'right'
                else:
                    action = 'left'
                thought_lines.append(f"I need to go get the key. Moving {action}.")

        elif not door_open:
            thought_lines.append("I have the key, but the door is closed.")
            if agent_pos == door_pos:
                action = 'open'
                thought_lines.append("I am at the door with the key. I will open it.")
            else:
                # Simple movement toward door
                ar, ac = agent_pos
                dr, dc = door_pos
                if ar < dr:
                    action = 'down'
                elif ar > dr:
                    action = 'up'
                elif ac < dc:
                    action = 'right'
                else:
                    action = 'left'
                thought_lines.append(f"I need to reach the door to open it. Moving {action}.")

        else:
            thought_lines.append("Door is open! Now going to the goal.")
            if agent_pos == goal_pos:
                action = None
                thought_lines.append("I have reached the goal!")
            else:
                # Simple movement toward goal
                ar, ac = agent_pos
                gr, gc = goal_pos
                if ar < gr:
                    action = 'down'
                elif ar > gr:
                    action = 'up'
                elif ac < gc:
                    action = 'right'
                else:
                    action = 'left'
                thought_lines.append(f"Heading to the goal. Moving {action}.")

        thought = " | ".join(thought_lines)
        return {'thought': thought, 'action': action}

    def act(self, action):
        if action is None:
            return None, 0.0, False, {'noop': True}
        return self.env.step(action)

    def run(self):
        # Initial observation
        obs = {
            'agent_pos': self.env.state.agent_pos,
            'has_key': self.env.state.has_key,
            'door_open': self.env.state.door_open,
            'key_pos': self.env.key_pos,
            'door_pos': self.env.door_pos,
            'goal_pos': self.env.goal_pos,
            'cell': self.env._cell_content(self.env.state.agent_pos)
        }

        done = False
        last_positions = deque(maxlen=6)

        print("🚀 Starting Simple Reason → Act → Observe Loop...\n")

        while not done and self.steps < self.max_steps:
            self.steps += 1
            reasoning = self.reason(obs)
            action = reasoning['action']
            thought = reasoning['thought']

            new_obs, reward, done, info = self.act(action)

            if new_obs is None:
                new_obs = obs.copy()

            self.trace.append({
                'step': self.steps,
                'thought': thought,
                'action': action,
                'observation': new_obs,
                'info': info
            })

            print(f"Step {self.steps}:")
            print(f"  Thought: {thought}")
            print(f"  Action : {action}")
            print(f"  Info   : {info}")
            self.env.render()
            print("-" * 50)

            last_positions.append(self.env.state.agent_pos)
            if len(last_positions) == last_positions.maxlen and len(set(last_positions)) == 1:
                print("Agent is stuck (not moving). Stopping.")
                break

            obs = new_obs

        if done:
            print(f" Goal reached in {self.steps} steps!")
        else:
            print(" Stopped (max steps or stuck).")

        return self.trace

In [10]:
# Define the grid
grid = [
    "##########",
    "#A       #",
    "#  #     #",
    "#  K  D  #",
    "#  #     #",
    "#       G#",
    "##########"
]

# Create environment and agent
env = GridWorld(grid)
print("Initial Environment:")
env.render()

agent = Agent(env, max_steps=100)
trace = agent.run()

# Simple Summary
print("\n=== SUMMARY TRACE ===")
for entry in trace[:10]:   # show first 10 steps only
    print(f"Step {entry['step']:2d} | Action: {entry['action']:6} | Thought: {entry['thought'].split(' | ')[0]}")

Initial Environment:
##########
#A       #
#  #     #
#  K  D  #
#  #     #
#       G#
##########
Has Key: False | Door Open: False

🚀 Starting Simple Reason → Act → Observe Loop...

Step 1:
  Thought: I don't have the key yet. | I need to go get the key. Moving down.
  Action : down
  Info   : {'moved': True}
##########
#        #
#A #     #
#  K  D  #
#  #     #
#       G#
##########
Has Key: False | Door Open: False

--------------------------------------------------
Step 2:
  Thought: I don't have the key yet. | I need to go get the key. Moving down.
  Action : down
  Info   : {'moved': True}
##########
#        #
#  #     #
#A K  D  #
#  #     #
#       G#
##########
Has Key: False | Door Open: False

--------------------------------------------------
Step 3:
  Thought: I don't have the key yet. | I need to go get the key. Moving right.
  Action : right
  Info   : {'moved': True}
##########
#        #
#  #     #
# AK  D  #
#  #     #
#       G#
##########
Has Key: False | Door Ope